# Flushing Ratio Workflow

## Preprocessing

### Image splitting

In [ ]:
# !pip install opencv-python numpy

import os
from pathlib import Path
from math import ceil
from concurrent.futures import ThreadPoolExecutor, as_completed
import cv2

# -------------------- SETTINGS --------------------
basic_folder  = Path(r"C:/path/to/your/folder/with/original/images")         #insert path to folder with your original images
output_folder = Path(r"C:/path/to/your/folder/where/folders/with/tiles")     #insert path to folder, where the splited images will be stored
mapping_file = Path(r"C:/path/to/control/file.txt")                         # insert path to the .txt file, where information will be stored
# or .csv if you prefer: Path(.../image_id_map.csv)

#---------------------------------------------------

part_width  = 600                                                          # basic tile width
part_height = 600                                                          # basic tile height
k_start = 1                                                                # resume from picture no. xxx (only if you don't run it from scratch)

# Lossless + fastest PNG write
PNG_COMPRESSION = 3                                                        # 0..9; 0 is fastest and STILL LOSSLESS
png_params = [cv2.IMWRITE_PNG_COMPRESSION, PNG_COMPRESSION]

# Optional: limit OpenCV internal threading to avoid oversubscription
cv2.setNumThreads(1)

# List images
img_names = sorted([
    p for p in basic_folder.iterdir()
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".JPG"}
])

k_end = len(img_names)

print(f"Found {k_end} image files in: {basic_folder}")
if k_end == 0:
    raise RuntimeError("No images found. Check folder path and extensions.")

# --- WRITE ID <-> FILENAME MAP ONCE ---
mapping_file.parent.mkdir(parents=True, exist_ok=True)

with open(mapping_file, "w", encoding="utf-8") as f:
    f.write("image_id\tfilename\tfull_path\n")  # header
    for idx, p in enumerate(img_names, start=1):
        f.write(f"{idx}\t{p.name}\t{str(p)}\n")

print("Saved mapping to:", mapping_file)
# --------------------------------------------

print(f"Found {k_end} image files in: {basic_folder}")
if k_end == 0:
    raise RuntimeError("No images found. Check folder path and extensions.")

print("First file:", img_names[0].name)
print("Last  file:", img_names[-1].name)

def cut_and_save_cv(img, out_dir: Path, k: int, x_offset: int, y_offset: int, tag: str):
    height, width = img.shape[:2]

    effective_w = width - x_offset
    effective_h = height - y_offset
    if effective_w <= 0 or effective_h <= 0:
        return

    num_parts_width  = ceil(effective_w / part_width)
    num_parts_height = ceil(effective_h / part_height)

    for i in range(1, num_parts_height + 1):
        y0 = (i - 1) * part_height + y_offset
        h = min(part_height, height - y0)
        if h <= 0:
            continue

        for j in range(1, num_parts_width + 1):
            x0 = (j - 1) * part_width + x_offset
            w = min(part_width, width - x0)
            if w <= 0:
                continue

            tile = img[y0:y0 + h, x0:x0 + w]
            out_name = f"{k}_{tag}_{i}_{j}.png"
            cv2.imwrite(str(out_dir / out_name), tile, png_params)

def process_one(k: int):
    "Process a single image index k (1-based). Returns a short status string."
    try:
        img_path = img_names[k - 1]
        out_dir = output_folder / str(k)
        out_dir.mkdir(parents=True, exist_ok=True)

        # Preserve channels/bit-depth when possible
        img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
        if img is None:
            return f"SKIP unreadable: k={k} {img_path.name}"

        cut_and_save_cv(img, out_dir, k, 0,   0,   "part")
        cut_and_save_cv(img, out_dir, k, 400, 0,   "400_right")
        cut_and_save_cv(img, out_dir, k, 200, 0,   "200_right")
        cut_and_save_cv(img, out_dir, k, 0,   400, "400_down")
        cut_and_save_cv(img, out_dir, k, 0,   200, "200_down")

        return f"OK k={k} {img_path.name}"
    except Exception as e:
        return f"ERR k={k}: {type(e).__name__}: {e}"

if k_start < 1 or k_start > k_end:
    raise ValueError(f"k_start must be in [1, {k_end}]")

# Threads: for heavy disk writing, 4–8 is usually best. Too high can slow/crash.
workers = min(8, max(4, (os.cpu_count() or 4) // 2))

jobs = list(range(k_start, k_end + 1))
print(f"Resuming at k={k_start}, ending at k={k_end} ({len(jobs)} images), workers={workers}")

done = 0
with ThreadPoolExecutor(max_workers=workers) as ex:
    futures = [ex.submit(process_one, k) for k in jobs]
    for fut in as_completed(futures):
        done += 1
        if done % 10 == 0:
            print(f"{done}/{len(jobs)}  {fut.result()}")

print("Done.")


## Running model inference

In [7]:
# control of GPU connection
!nvidia-smi

Mon Feb 16 16:09:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.08                 Driver Version: 581.08         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   56C    P8              4W /   15W |     865MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### running inference YOLOv11 to json masks

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import json
import numpy as np
import cv2
from pycocotools import mask as mask_utils

# ---------- config ----------
model_path = Path(r"C:/path/to/weights.pt")                                 # path to the model weights.pt (YOLO11_NorwaySpruce_SpruceBud.pt)
data_root  = Path(r"C:/path/to/folder/withsplitted/files")                  # = output_folder from Image splitting
out_root   = Path(r"C:/path/to/folder/where/jsons/will/be/saved")           # in this folder, the resulting jsons from inference will be saved

BATCH_SIZE = 12                                                             # good starting point for 4GB VRAM (use 4 if OOM)

# ---------- helpers ----------
def build_union_masks_from_results(result):
    H, W = result.orig_shape

    NS = np.zeros((H, W), np.uint8)
    SB = np.zeros((H, W), np.uint8)
    BG = np.zeros((H, W), np.uint8)

    if result.masks is None:
        return NS, SB, BG

    masks = (result.masks.data > 0.5).cpu().numpy().astype(np.uint8)
    classes = result.boxes.cls.cpu().numpy().astype(int)
    id_to_name = result.names

    for i, cls_id in enumerate(classes):
        m = masks[i]
        if m.shape != (H, W):
            m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)

        name = id_to_name[int(cls_id)]
        if name == "norway-spruce":
            NS |= m
        elif name == "spruce-bud":
            SB |= m
        elif name == "background":
            BG |= m

    return NS, SB, BG


def compute_logical_masks(NS, SB, BG):
    FG = ((NS | SB) & (1 - BG)).astype(np.uint8)
    RB = (SB & (1 - BG)).astype(np.uint8)
    return FG, RB


def coco_json_from_binary_mask(mask01, file_name, category_id, category_name):
    rle = mask_utils.encode(np.asfortranarray(mask01.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")

    return {
        "images": [{"id": 1, "file_name": file_name}],
        "categories": [{"id": category_id, "name": category_name, "supercategory": "none"}],
        "annotations": [{
            "id": 1,
            "image_id": 1,
            "category_id": category_id,
            "segmentation": rle,
            "bbox": list(map(float, mask_utils.toBbox(rle).tolist())),
            "area": float(mask_utils.area(rle)),
            "iscrowd": 0,
        }]
    }


# ---------- run ----------
model = YOLO(model_path)

# automatically find all subfolders inside data_root
folders = sorted([p for p in data_root.iterdir() if p.is_dir()])

for folder in folders:
    folder_id = folder.name  # keeps original name
    images = sorted(folder.glob("*.png"))

    if not images:
        print(f"Done folder {folder_id}")
        continue

    out_dir = out_root / folder_id
    out_dir.mkdir(parents=True, exist_ok=True)

    for i in range(0, len(images), BATCH_SIZE):
        batch_paths = images[i:i + BATCH_SIZE]
        batch_strs = [str(p) for p in batch_paths]

        results = model.predict(
            batch_strs,
            device=0,
            batch=BATCH_SIZE,
            imgsz=640,
            half=True,
            verbose=False
        )

        for img_path, r in zip(batch_paths, results):
            stem = img_path.stem
            rel_name = str(img_path.relative_to(data_root)).replace("\\", "/")

            NS, SB, BG = build_union_masks_from_results(r)
            FG, RB = compute_logical_masks(NS, SB, BG)

            FG_json = coco_json_from_binary_mask(FG, rel_name, 1, "FG")
            RB_json = coco_json_from_binary_mask(RB, rel_name, 2, "RB")

            with open(out_dir / f"{stem}_FG.json", "w", encoding="utf-8") as f:
                json.dump(FG_json, f, ensure_ascii=False)

            with open(out_dir / f"{stem}_RB.json", "w", encoding="utf-8") as f:
                json.dump(RB_json, f, ensure_ascii=False)

    print(f"Done folder {folder_id}")

## Postprocessing

### Stitching jsons together and overlaying

In [17]:
#stitching bez znalosti velikosti originálního obrázku
import os
import re
from pathlib import Path
import numpy as np
from pycocotools import mask as mask_utils

# --- fastest json loader available ---
try:
    import orjson
    def load_json(p): return orjson.loads(Path(p).read_bytes())
    def dump_json(obj, p): Path(p).write_bytes(orjson.dumps(obj))
except Exception:
    import json
    def load_json(p):
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    def dump_json(obj, p):
        with open(p, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False)

# ----- parameters -----
TILE_STEP = 600

PAT = re.compile(
    r"^(?P<x>\d+)"
    r"_(?:(?P<shift>\d+)_(?P<dir>right|down)|(?P<base>part))"
    r"_(?P<i>\d+)_(?P<j>\d+)"
    r"_(?P<kind>FG|RB)\.json$",
    re.IGNORECASE
)

def tile_origin(shift, direction, i, j):
    """Top-left (y, x) in full image for tile (i,j) with optional shift."""
    i = int(i); j = int(j)
    x0 = 0; y0 = 0
    if shift and direction:
        s = int(shift)
        if direction == "right":
            x0 = s
        elif direction == "down":
            y0 = s
    x = x0 + (j - 1) * TILE_STEP
    y = y0 + (i - 1) * TILE_STEP
    return y, x

def coco_from_fullmask(mask01, file_name, category_id, category_name):
    rle = mask_utils.encode(np.asfortranarray(mask01.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")
    return {
        "images": [{"id": 1, "file_name": file_name, "height": int(mask01.shape[0]), "width": int(mask01.shape[1])}],
        "categories": [{"id": int(category_id), "name": category_name, "supercategory": "none"}],
        "annotations": [{
            "id": 1, "image_id": 1, "category_id": int(category_id),
            "segmentation": rle,
            "bbox": list(map(float, mask_utils.toBbox(rle).tolist())),
            "area": float(mask_utils.area(rle)),
            "iscrowd": 0
        }]
    }


def stitch_one_x(x_id: str, tiles_root: Path, out_root: Path):
    prefix = f"{x_id}_"

    # -------- pass 1: infer FULL_H, FULL_W from all tile RLE sizes + positions --------
    max_y = 0
    max_x = 0
    matched_files = 0

    for entry in os.scandir(tiles_root):
        if not entry.is_file():
            continue
        name = entry.name
        if not name.startswith(prefix) or not name.endswith(".json"):
            continue

        m = PAT.match(name)
        if not m:
            continue

        coco = load_json(entry.path)
        anns = coco.get("annotations", [])
        if not anns:
            continue

        ann0 = anns[0]
        rle = ann0.get("segmentation", None)
        if not rle or "size" not in rle:
            continue

        th, tw = rle["size"]  # tile H,W (exact, even for edge tiles)

        shift = m.group("shift")
        direction = (m.group("dir") or "").lower()
        i = m.group("i")
        j = m.group("j")
        y, x = tile_origin(shift, direction, i, j)

        max_y = max(max_y, y + int(th))
        max_x = max(max_x, x + int(tw))
        matched_files += 1

    if matched_files == 0 or max_y <= 0 or max_x <= 0:
        raise RuntimeError(f"No usable tiles found for x_id={x_id} in {tiles_root}")

    FULL_H = int(max_y)
    FULL_W = int(max_x)

    # Allocate full masks dynamically
    FG_full = np.zeros((FULL_H, FULL_W), dtype=np.uint8)
    RB_full = np.zeros((FULL_H, FULL_W), dtype=np.uint8)

    # -------- pass 2: stitch masks (decode only if area>0) --------
    for entry in os.scandir(tiles_root):
        if not entry.is_file():
            continue
        name = entry.name
        if not name.startswith(prefix) or not name.endswith(".json"):
            continue

        m = PAT.match(name)
        if not m:
            continue

        kind = m.group("kind").upper()
        shift = m.group("shift")
        direction = (m.group("dir") or "").lower()
        i = m.group("i")
        j = m.group("j")

        coco = load_json(entry.path)
        anns = coco.get("annotations", [])
        if not anns:
            continue
        ann0 = anns[0]

        rle = ann0.get("segmentation", None)
        if not rle or "size" not in rle:
            continue

        th, tw = rle["size"]

        # For stitching pixels, you can skip empty tiles
        if ann0.get("area", 0) == 0:
            continue

        tile = mask_utils.decode(rle)
        if tile.ndim == 3:
            tile = tile[:, :, 0]
        tile = (tile > 0).astype(np.uint8)

        # Ensure consistent with RLE size
        if tile.shape != (int(th), int(tw)):
            th, tw = tile.shape

        y, x = tile_origin(shift, direction, i, j)

        # Clip placement
        y1 = max(0, y)
        x1 = max(0, x)
        y2 = min(FULL_H, y + int(th))
        x2 = min(FULL_W, x + int(tw))
        if y1 >= y2 or x1 >= x2:
            continue

        ty1 = y1 - y
        tx1 = x1 - x
        ty2 = ty1 + (y2 - y1)
        tx2 = tx1 + (x2 - x1)

        if kind == "FG":
            FG_full[y1:y2, x1:x2] |= tile[ty1:ty2, tx1:tx2]
        else:
            RB_full[y1:y2, x1:x2] |= tile[ty1:ty2, tx1:tx2]

    out_root.mkdir(parents=True, exist_ok=True)
    dump_json(coco_from_fullmask(FG_full, f"{x_id}.png", 1, "FG"), out_root / f"{x_id}_FG.json")
    dump_json(coco_from_fullmask(RB_full, f"{x_id}.png", 2, "RB"), out_root / f"{x_id}_RB.json")


In [18]:
from pathlib import Path

base_tiles = Path(r"C:/path/to/folder/with/jsons")                          # = out_root from running inference YOLOv11 to json masks
base_out   = Path(r"C:/path/to/folder/where/outputs/will/be/store")         # in this folder stitched outputs from json will be stored

for tiles_root in sorted(p for p in base_tiles.iterdir() if p.is_dir()):
    folder_id = tiles_root.name  # např. "1", "2", ...
    out_root = base_out / folder_id
    out_root.mkdir(parents=True, exist_ok=True)

    stitch_one_x(folder_id, tiles_root, out_root)


### json rotation

In [ ]:
# inference has turned the jsons about 90° counter-clockwise
   # we need to turn the jsons back

from pathlib import Path
import json
import numpy as np
from pycocotools import mask as mask_utils


def rotate_mask_90_cw(mask01: np.ndarray) -> np.ndarray:
    return np.rot90(mask01, k=-1).astype(np.uint8)


def rotate_rle_90_cw(rle: dict) -> dict:
    rle_in = dict(rle)

    # ensure counts are bytes for pycocotools
    if isinstance(rle_in["counts"], str):
        rle_in["counts"] = rle_in["counts"].encode("ascii")

    mask = mask_utils.decode(rle_in)
    mask_rot = rotate_mask_90_cw(mask)

    rle_out = mask_utils.encode(np.asfortranarray(mask_rot))
    rle_out["counts"] = rle_out["counts"].decode("ascii")

    return rle_out


def rotate_coco_json_inplace(json_path: Path):

    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    # rotate annotations
    for ann in coco.get("annotations", []):
        seg = ann.get("segmentation")

        if isinstance(seg, dict) and "counts" in seg:
            new_rle = rotate_rle_90_cw(seg)
            ann["segmentation"] = new_rle

            # recompute bbox + area
            tmp = dict(new_rle)
            tmp["counts"] = tmp["counts"].encode("ascii")

            ann["bbox"] = list(map(float, mask_utils.toBbox(tmp).tolist()))
            ann["area"] = float(mask_utils.area(tmp))
        else:
            raise ValueError(f"{json_path} does not contain RLE segmentation")

    # swap width/height
    for img in coco.get("images", []):
        if "width" in img and "height" in img:
            img["width"], img["height"] = img["height"], img["width"]

    # overwrite original file
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False)

    print("Fixed:", json_path.name)


def batch_fix(root: Path):
    json_files = sorted(root.rglob("*.json"))

    for jp in json_files:
        rotate_coco_json_inplace(jp)


if __name__ == "__main__":
    ROOT = Path(r"C:/path/to/folde/with/stitched/jsons")        # = base_out from Stitching jsons togehter and overlaying
    batch_fix(ROOT)



### cutout img from json

In [28]:
import json
from pathlib import Path
import cv2
import numpy as np
from pycocotools import mask as mask_utils

def cutout_png_from_coco_rle(
    image_path: str,
    coco_json_path: str,
    out_png_path: str,
    zero_rgb_where_transparent: bool = True,
):
    """
    Reads a COCO-style JSON with RLE (single or multiple annotations),
    unions masks, and saves a full-size RGBA PNG cutout:
      - RGB from original image where mask==1
      - alpha = 255 where mask==1, else 0 (transparent background)
    """
    image_path = Path(image_path)
    coco_json_path = Path(coco_json_path)
    out_png_path = Path(out_png_path)
    out_png_path.parent.mkdir(parents=True, exist_ok=True)

    img_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    H, W = img_bgr.shape[:2]

    with open(coco_json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    anns = coco.get("annotations", [])
    if not anns:
        # Save fully transparent PNG if you want, or raise
        out_rgba = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2BGRA)
        out_rgba[:, :, 3] = 0
        cv2.imwrite(str(out_png_path), out_rgba)
        return 0

    # Union all annotations (works for your single-annotation FG/RB JSONs too)
    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        rle = ann["segmentation"]
        m = mask_utils.decode(rle)
        if m.ndim == 3:
            m = m[:, :, 0]
        m = (m > 0).astype(np.uint8)

        # In case JSON mask size differs (shouldn't if encoded correctly)
        if m.shape != (H, W):
            m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)

        union = np.maximum(union, m)

    # Create RGBA cutout
    out_rgba = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2BGRA)
    out_rgba[:, :, 3] = (union * 255).astype(np.uint8)

    if zero_rgb_where_transparent:
        out_rgba[union == 0, 0:3] = 0  # avoids “ghost background” in some viewers

    cv2.imwrite(str(out_png_path), out_rgba)
    return int(union.sum())


In [ ]:

# fill the numbers of your images
for i in range(2106,2108):
    img_path = f"C:/path/to/your/images/IMGP{i}.JPG"                    # path to your original images
    stem = Path(img_path).stem

    derived_dir = Path(f"C:/path/to/your/jsons/{i-2105}")               # if needed, change the loop logic, fill the path to rotated jsons
    fg_json = derived_dir / f"{i-2105}_FG.json"
    rb_json = derived_dir / f"{i-2105}_RB.json"

    out_dir = Path(f"C:/path/to/your/output/cutouts")                   # path to your cutouts from original images
    out_dir.mkdir(parents=True, exist_ok=True)

    fg_png = out_dir / f"{stem}_FG_cutout.png"
    rb_png = out_dir / f"{stem}_RB_cutout.png"

    fg_pixels = cutout_png_from_coco_rle(img_path, fg_json, fg_png)
    rb_pixels = cutout_png_from_coco_rle(img_path, rb_json, rb_png)

    print("cutout number ",i, "saved")
    #print("FG cutout saved:", fg_png, "pixels kept:", fg_pixels)
    #print("RB cutout saved:", rb_png, "pixels kept:", rb_pixels)

### thresholding

In [36]:
import json
from pathlib import Path
import cv2
import numpy as np
from pycocotools import mask as mask_utils

def coco_json_from_mask(mask01: np.ndarray, file_name: str, category_id: int, category_name: str):
    rle = mask_utils.encode(np.asfortranarray(mask01.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")

    return {
        "images": [{"id": 1, "file_name": file_name, "height": int(mask01.shape[0]), "width": int(mask01.shape[1])}],
        "categories": [{"id": int(category_id), "name": category_name, "supercategory": "none"}],
        "annotations": [{
            "id": 1,
            "image_id": 1,
            "category_id": int(category_id),
            "segmentation": rle,
            "bbox": list(map(float, mask_utils.toBbox(rle).tolist())),
            "area": float(mask_utils.area(rle)),
            "iscrowd": 0
        }]
    }

def threshold_cutout_rgba_to_mask(
    rgba_path: str,
    L_min=1, L_max=254,
    a_min=1, a_max=125,
    b_min=130, b_max=254,
):
    """
    Reads an RGBA cutout PNG (transparent background).
    Returns binary mask (H,W) uint8 {0,1} of pixels that pass the LAB thresholds,
    restricted to alpha > 0.
    """
    rgba = cv2.imread(str(rgba_path), cv2.IMREAD_UNCHANGED)
    if rgba is None:
        raise FileNotFoundError(f"Cannot read: {rgba_path}")
    if rgba.ndim != 3 or rgba.shape[2] != 4:
        raise ValueError(f"Expected RGBA image with 4 channels, got shape {rgba.shape}")

    bgr = rgba[:, :, :3]
    alpha = rgba[:, :, 3]

    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    L = lab[:, :, 0]
    a = lab[:, :, 1]
    b = lab[:, :, 2]

    keep = (
        (alpha > 0) &
        (L >= L_min) & (L <= L_max) &
        (a >= a_min) & (a <= a_max) &
        (b >= b_min) & (b <= b_max)
    )

    return keep.astype(np.uint8)

def threshold_cutout_to_json(
    rgba_cutout_path: str,
    out_json_path: str,
    category_id: int = 1,
    category_name: str = "thresholded",
    **thr
):
    mask01 = threshold_cutout_rgba_to_mask(rgba_cutout_path, **thr)

    out_json_path = Path(out_json_path)
    out_json_path.parent.mkdir(parents=True, exist_ok=True)

    coco = coco_json_from_mask(mask01, file_name=Path(rgba_cutout_path).name, category_id=category_id, category_name=category_name)

    with open(out_json_path, "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False, indent=2)

    return int(mask01.sum())  # pixels kept


#### FG thresholding

In [ ]:
for i in range(2106,2108):
    rgba_cutout = f"C:/path/to/your/cutouts/IMGP{i}_FG_cutout.png"                                    # fill the path
    out_json    = f"C:/path/to/folde/where/final/jsons/will/be/stored/IMGP{i}_FG_thresholded.json"    # fill the path   

    px = threshold_cutout_to_json(
        rgba_cutout,
        out_json,
        category_id=1,
        category_name="FG_thresholded",
        L_min=1, L_max=254,                           # you can adjust the thresholding
        a_min=1, a_max=125,                           # you can adjust the thresholding
        b_min=130, b_max=254                          # you can adjust the thresholding 
    )

    print("Saved:", out_json, "pixels kept:", px)


#### RB thresholding

In [ ]:
for i in range(1,678):
    rgba_cutout = f"C:/path/to/your/cutouts/IMGP{i}_RB_cutout.png"                                    # fill the path
    out_json    = f"C:/path/to/folde/where/final/jsons/will/be/stored/IMGP{i}_RB_thresholded.json"    # fill the path  

    px = threshold_cutout_to_json(
        rgba_cutout,
        out_json,
        category_id=1,
        category_name="RB_thresholded",
        L_min=1, L_max=254,                           # you can adjust the thresholding
        a_min=1, a_max=125,                           # you can adjust the thresholding
        b_min=130, b_max=254                          # you can adjust the thresholding 
    )

    print("Saved:", out_json, "pixels kept:", px)

### Pixel count record

In [ ]:
from pathlib import Path
import json

# ---- paths ----
json_dir = Path(r"C:\Users\jirka\Desktop\pokus\cutouts_from_json")                      # contains x_FG*.json and x_RB*.json
out_txt  = Path(r"C:\Users\jirka\Desktop\pokus\pixel_counts_L25.txt")

# ---- helper ----
def get_area_from_coco(json_path: Path) -> int:
    if not json_path.exists():
        return 0
    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)
    anns = coco.get("annotations", [])
    if not anns:
        return 0
    return int(anns[0].get("area", 0))

# ---- write table ----
with open(out_txt, "w", encoding="utf-8") as f:
    f.write("image_id\tFG_pixels\tRB_pixels\n")

    for img_id in range(2106, 2108):
        img_id_str = str(img_id)

        fg_json = json_dir / f"IMGP{i}_FG_thresholded.json"
        rb_json = json_dir / f"IMGP{i}_RB_thresholded.json"

        fg_pixels = get_area_from_coco(fg_json)
        rb_pixels = get_area_from_coco(rb_json)

        f.write(f"{img_id}\t{fg_pixels}\t{rb_pixels}\n")

print("Saved:", out_txt)


### phenomcams indices RCC and GCC

In [ ]:
# pip install opencv-python numpy pycocotools orjson

import csv
from pathlib import Path
import numpy as np
import cv2
from pycocotools import mask as mask_utils

# ---------- fast json loader ----------
try:
    import orjson
    def load_json(p: Path):
        return orjson.loads(p.read_bytes())
except Exception:
    import json
    def load_json(p: Path):
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)

# ---------- helpers ----------
def decode_coco_rle_mask(coco_path: Path) -> np.ndarray:
    """
    Decode COCO RLE mask JSON into uint8 mask (0/1).
    Unions all annotations if multiple.
    """
    coco = load_json(coco_path)
    anns = coco.get("annotations", [])
    if not anns:
        raise ValueError(f"No annotations in {coco_path.name}")

    out = None
    for ann in anns:
        rle = ann.get("segmentation", None)
        if not rle or "size" not in rle:
            continue

        # pycocotools expects counts as bytes
        if isinstance(rle.get("counts", None), str):
            rle = dict(rle)
            rle["counts"] = rle["counts"].encode("ascii")

        m = mask_utils.decode(rle)
        if m.ndim == 3:
            m = m[:, :, 0]
        m = (m > 0).astype(np.uint8)

        out = m if out is None else (out | m)

    if out is None:
        raise ValueError(f"No usable RLE in {coco_path.name}")

    return out

def id_from_filename(p: Path):
    """
    Extract leading integer ID from filename stem.
    Examples: '123.jpg' -> 123, '123_anything.png' -> 123
    """
    s = p.stem
    num = ""
    for ch in s:
        if ch.isdigit():
            num += ch
        else:
            break
    return int(num) if num else None

def compute_gcc_image_with_mask(
    img_path: Path,
    mask_json_path: Path,
    black_thresh: int = 0,   # ignore pixels that are too black (0 = ignore pure black only)
    use_alpha: bool = True,  # if image has alpha, require alpha > 0
):
    """
    Memory-safe GCC/RCC:
    - no float64 conversions
    - no cv2.split copies
    - sums computed in uint64 with 'where=valid'
    """
    img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
    if img is None:
        return {"ok": False, "error": "unreadable_image"}

    if img.ndim == 2:
        return {"ok": False, "error": "grayscale_image"}

    if img.shape[2] < 3:
        return {"ok": False, "error": "not_rgb_image"}

    # BGR(A) views (no copies)
    b = img[..., 0]
    g = img[..., 1]
    r = img[..., 2]
    a = img[..., 3] if (img.shape[2] == 4) else None

    # Decode ROI mask
    try:
        mask01 = decode_coco_rle_mask(mask_json_path).astype(bool)
    except Exception as e:
        return {"ok": False, "error": f"mask_decode_error: {type(e).__name__}: {e}"}

    h, w = r.shape[:2]
    if mask01.shape == (w, h):
        mask01 = np.rot90(mask01, k=-1)  # try -1; if wrong, use +1
    if mask01.shape != (h, w):
        return {"ok": False, "error": f"mask_size_mismatch mask={mask01.shape} img={(h,w)}"}

    # Valid pixels start with ROI
    valid = mask01

    # Optionally also exclude black-ish pixels
    if black_thresh is not None:
        valid = valid & ((r > black_thresh) | (g > black_thresh) | (b > black_thresh))

    # Optionally exclude transparent pixels
    if use_alpha and (a is not None):
        valid = valid & (a > 0)

    n_valid = int(valid.sum())
    if n_valid == 0:
        return {"ok": True, "gcc": np.nan, "rcc": np.nan, "n_valid": 0, "error": ""}

    # Sums in uint64 (safe for big images)
    sum_r = int(np.sum(r, where=valid, dtype=np.uint64))
    sum_g = int(np.sum(g, where=valid, dtype=np.uint64))
    sum_b = int(np.sum(b, where=valid, dtype=np.uint64))

    denom = sum_r + sum_g + sum_b
    if denom <= 0:
        return {"ok": True, "gcc": np.nan, "rcc": np.nan, "n_valid": n_valid, "error": ""}

    gcc = sum_g / denom
    rcc = sum_r / denom

    return {
        "ok": True,
        "gcc": float(gcc),
        "rcc": float(rcc),
        "n_valid": n_valid,
        "error": ""
    }

def compute_folder_gcc(
    images_dir: Path,
    masks_dir: Path,
    out_csv: Path,
    mask_suffix: str = "_FG_thresholded.json",
    masks_in_subfolders: bool = False,  # True if masks are in masks_dir/{id}/{id}_suffix
    black_thresh: int = 0,
    use_alpha: bool = True,
):
    images_dir = Path(images_dir)
    masks_dir = Path(masks_dir)
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
    imgs = [p for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in exts]

    # numeric sort by id when possible
    imgs = sorted(imgs, key=lambda p: (id_from_filename(p) is None, id_from_filename(p) or 10**12, p.name))

    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["image_id", "image_file", "mask_file", "gcc", "rcc", "n_valid", "ok", "error"])

        for idx, img_path in enumerate(imgs, start=1):
            img_id = id_from_filename(img_path)
            if img_id is None:
                w.writerow(["", img_path.name, "", "", "", "", False, "cannot_parse_id"])
                continue

            if masks_in_subfolders:
                mask_path = masks_dir / str(img_id) / f"{img_id}{mask_suffix}"
            else:
                mask_path = masks_dir / f"{img_id}{mask_suffix}"

            if not mask_path.exists():
                w.writerow([img_id, img_path.name, mask_path.name, "", "", "", False, "mask_missing"])
                continue

            res = compute_gcc_image_with_mask(
                img_path=img_path,
                mask_json_path=mask_path,
                black_thresh=black_thresh,
                use_alpha=use_alpha,
            )

            w.writerow([
                img_id,
                img_path.name,
                mask_path.name,
                res.get("gcc", ""),
                res.get("rcc", ""),
                res.get("n_valid", ""),
                res.get("ok", False),
                res.get("error", ""),
            ])

            if idx % 50 == 0:
                print(f"{idx}/{len(imgs)} done...")

    print("Saved:", out_csv)

# ------------------- RUN HERE -------------------
images_dir = Path(r"C:/path/to/your/orig/images")                                  # fill the path
masks_dir  = Path(r"C:/path/to/your/cutouts_from_json")                            # where 123_FG_thresholded.json lives
out_csv    = Path(r"C:/path/to/your/csv/output/RCC_GCC_phenocam_values.csv")       # fill the path

compute_folder_gcc(
    images_dir=images_dir,
    masks_dir=masks_dir,
    out_csv=out_csv,
    mask_suffix="_FG_thresholded.json",
    masks_in_subfolders=False,                                                     # set True if your masks are in .../cutouts_from_json/{id}/...
    black_thresh=0,                                                                # try 5 or 10 if “almost black” background leaks in
    use_alpha=True
)
